# 장르+줄거리

### 전처리

In [1]:
!pip install konlpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 54.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.3/465.3 kB 47.4 MB/s eta 0:00:00


In [2]:
!pip install nltk

In [3]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.6 MB/s eta 0:00:00
  Created wheel for sentence_transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=edcbe94018652af754fb1bf9d74c4b80190b2d6760b9e061b4bc4f7cd29794c6
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence_transformers


In [5]:
import torch
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
from urllib.request import urlopen
import os
import re
from konlpy.tag import Okt, Kkma, Komoran
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import networkx
from tqdm.autonotebook import tqdm

import seaborn as sns
import matplotlib.pyplot as plt

from gensim.models import word2vec
import itertools

In [7]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

.gitattributes:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

In [8]:
df = pd.read_excel('../data/raw/theater_top77.xlsx')
df = df.drop(df.columns[0], axis=1)
# df = df.drop(df.columns[2], axis=1)
df.head(3)

,Title,Text_clear_num,연극 이름,공연소개,줄거리,장르
0,연극 〈운빨로맨스〉- 대학로,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트목숨이 걸린 중...,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,NaN,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...,"로맨스, 코믹, 공감/힐링"
1,(리얼타임 코믹연극) 택시안에서 - 서울,웃음 감동 사랑이 시작되는 리얼타엄 코믹연극 공연시간 투 콩감백배 다양한에피소드와 ...,(리얼타임 코믹연극) 택시안에서 - 부산,NaN,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,"코믹, 로맨스, 공감/힐링"
2,(코믹연극) 달동네-부산,전사 오픈런 부족 초대합니다 개스 테 그를 코 뻔 부 토사 사으소시 서사으 냈 이거...,(코믹연극) 달동네-부산,NaN,"정음의 아버지 경민은 행정착오로 인해 월남전 참전 중 전사자 처리가 되고,........","코믹, 감동, 드라마"


In [9]:
df1 = df[['Title', '줄거리', '장르']]
df1.head(3)

,Title,줄거리,장르
0,연극 〈운빨로맨스〉- 대학로,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...,"로맨스, 코믹, 공감/힐링"
1,(리얼타임 코믹연극) 택시안에서 - 서울,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,"코믹, 로맨스, 공감/힐링"
2,(코믹연극) 달동네-부산,"정음의 아버지 경민은 행정착오로 인해 월남전 참전 중 전사자 처리가 되고,........","코믹, 감동, 드라마"


In [10]:
df1['장르']=df1['장르'].str.split(',')
df1['장르'] = df1['장르'].apply(lambda x : (' ').join(x))
df1.head(3)

<ipython-input-10-64c876e4423b>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['장르']=df1['장르'].str.split(',')
<ipython-input-10-64c876e4423b>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['장르'] = df1['장르'].apply(lambda x : (' ').join(x))


,Title,줄거리,장르
0,연극 〈운빨로맨스〉- 대학로,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...,로맨스 코믹 공감/힐링
1,(리얼타임 코믹연극) 택시안에서 - 서울,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,코믹 로맨스 공감/힐링
2,(코믹연극) 달동네-부산,"정음의 아버지 경민은 행정착오로 인해 월남전 참전 중 전사자 처리가 되고,........",코믹 감동 드라마


In [11]:
df1['줄거리'] = df1['줄거리'].apply(lambda x: re.sub(r'[^가-힣a-zA-Z\s]', '', str(x)))
df1.head(3)

<ipython-input-11-7b1018866824>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['줄거리'] = df1['줄거리'].apply(lambda x: re.sub(r'[^가-힣a-zA-Z\s]', '', str(x)))


,Title,줄거리,장르
0,연극 〈운빨로맨스〉- 대학로,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...,로맨스 코믹 공감/힐링
1,(리얼타임 코믹연극) 택시안에서 - 서울,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남\n남...,코믹 로맨스 공감/힐링
2,(코믹연극) 달동네-부산,정음의 아버지 경민은 행정착오로 인해 월남전 참전 중 전사자 처리가 되고\n잘못된 ...,코믹 감동 드라마


In [12]:
# 줄거리 전체 + 장르 합쳐서 새로운 열 생성
df_data = df1.drop(df1[(df1['장르']=='') | (df1['줄거리'] =='')].index)
df_data['genres_keywords'] = df1['장르'] + " " +(df1['줄거리'])

In [13]:
df = pd.read_excel("../data/raw/musical_top77.xlsx")
df = df.drop(df.columns[0], axis=1)
# df = df.drop(df.columns[2], axis=1)
df.head()

,Title,Text_clear_num,뮤지컬 이름,공연소개,줄거리,장르
0,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",인기 우리의 작은 외침이 세상을 바꾼파 우리의 작은 외침이 세상을 바꾼다 우리들의 ...,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",NaN,시조'가 국가이념인 상상 속의 '조선'\n삶의 고단함과 역경을 시조 속에 담아 훌훌...,"역사극, 판타지"
1,2023 창작뮤지컬어워드 NEXT,창작뮤지컬어워드 충무마트센터가 가능성 있는 참작뮤지컬 발굴을 플랫품인 창작뮤지컬머워...,2023 창작뮤지컬어워드 NEXT,None,헌책방을 운영하는 에이미는 여느 날과 다름없이 책들에게 인사를 건네며 하루를 시작한...,드라마
2,2023 최현우 Answer - 성남,적 서니 정남아트센터 오페라하우스 이하늘이엔티 빅탑엔터테인언트 지라온플레이 년간의 ...,2023 최현우 Answer - 성남,NaN,"마술사로 걸어온 27년\n2,700여회 공연, 200만 명 관객 돌파\n세계가 인정...","마술, 판타지"
3,2023~24 최현우 Answer,마포아트센터 아트홀 맥 외는 이라을레이 이라오 저아아개 년간의 마술 노하우를 집대성...,2023~24 최현우 Answer,NaN,"마술사로 걸어온 27년\n2,700여회 공연, 200만 명 관객 돌파\n세계가 인정...","마술, 판타지"
4,2023송승환의 오리지널 난타(대전),렵 노역 톨 한받때약교야트을좌적매치도 슈 태빼 도시 해외공연투어 장기공연의 힘 오직...,NaN,NaN,세 명의 요리사가 하루 일과를 시작한다. 즐겁게 요리를 준비하는 동안 지배인의 깜짝...,"코믹, 액션"


In [14]:
df1 = df[['Title', '줄거리', '장르']]
df1.head(3)

,Title,줄거리,장르
0,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",시조'가 국가이념인 상상 속의 '조선'\n삶의 고단함과 역경을 시조 속에 담아 훌훌...,"역사극, 판타지"
1,2023 창작뮤지컬어워드 NEXT,헌책방을 운영하는 에이미는 여느 날과 다름없이 책들에게 인사를 건네며 하루를 시작한...,드라마
2,2023 최현우 Answer - 성남,"마술사로 걸어온 27년\n2,700여회 공연, 200만 명 관객 돌파\n세계가 인정...","마술, 판타지"


In [15]:
df1['장르']=df1['장르'].str.split(',')
df1['장르'] = df1['장르'].apply(lambda x : (' ').join(x))

df1['줄거리'] = df1['줄거리'].apply(lambda x: re.sub(r'[^가-힣a-zA-Z\s]', '', str(x)))
df_data = df1.drop(df1[(df1['장르']=='') | (df1['줄거리'] =='')].index)
df_data['genres_keywords'] = df1['장르'] + " " +(df1['줄거리'])

<ipython-input-15-7bf060d3e4ab>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['장르']=df1['장르'].str.split(',')
<ipython-input-15-7bf060d3e4ab>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['장르'] = df1['장르'].apply(lambda x : (' ').join(x))
<ipython-input-15-7bf060d3e4ab>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/s

### 장르 + 줄거리 유사도

In [16]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')
embedder.max_seq_len = 128

titles = df_data['Title'].tolist()
stories = df_data['genres_keywords'].tolist()

# Fit model to corpus qnd push to GPU
story_embeddings = embedder.encode(stories, convert_to_tensor=True)

In [17]:
def semantic_search(input_data):
  title_list = []
  story_list = []
  score_list = []

  results = pd.DataFrame()
  # Find the closest 5 stories of the corpus for each query sentence based on cosine similarity
  top_k = min(6, len(story_embeddings))

  # If the input is too short to be a story or its not in the dataset
  if len(input_data) < 20 and input_data not in titles:
    print('Title Not Found')

  # If input is in the dataset
  elif input_data in titles:
    query_embeddings = embedder.encode(str(df_data[df_data['Title'] == input_data]['genres_keywords'])[5:-33], convert_to_tensor=True)

    # Use cosine-similarity and torch.topk to find the highest 5 scores
    cos_scores = util.pytorch_cos_sim(query_embeddings, story_embeddings)[0]
    top_results = torch.topk(cos_scores, k=65)

    input_data_2 = input_data.replace('.', '.\n')
    print("\n\n======================")
    print("    TOP RESULTS")
    print("======================\n")

    # For score, index in torch.topk(cos_scores, k=top_k) use index  locator for feature lists
    # push score to cpu and convert to 1D array
    for score, idx in zip(top_results[0], top_results[1]):
      title_list.append(titles[idx])
      story_list.append(stories[idx])
      score_list.append(score.cpu().numpy().flatten())

    results['Title'] = title_list
    results['Story'] = story_list
    results['Score'] = score_list
    # return dictionary
    return results.iloc[1:, :]

  # If the input is long enough to be a story which is in the dataset
  elif len(input_data) > 20 and input_data not in titles:
    # Find the closest 5 stories of the corpus for each query sentence based on cosine similarity
    query_embeddings = embedder.encode(input_data, convert_to_tensor=True)

    # Use cosine-similarity and torch.topk to find the highest 5 scores
    cos_scores = util.pytorch_cos_sim(query_embeddings, story_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)

    input_data = input_data.replace('.', '.\n')
    print(input_data)
    print("\n\n======================")
    print("    TOP RESULTS")
    print("======================\n")

    # For score, index in torch.topk(cos_scores, k=top_k) use index  locator for feature lists
    # push score to cpu and convert to 1D array
    for score, idx in zip(top_results[0], top_results[1]):
      title_list.append(titles[idx])
      story_list.append(stories[idx])
      score_list.append(score.cpu().numpy().flatten())

    # Push results to dictionary columns
    results['Title'] = title_list
    results['Story'] = story_list
    results['Score'] = score_list
    # return dictionary
    return results

In [18]:
### 뮤지컬 장르+줄거리 임베딩
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')
embedder.max_seq_len = 128

titles = df_data['Title'].tolist()
stories = df_data['genres_keywords'].tolist()

# Fit model to corpus qnd push to GPU
story_embeddings = embedder.encode(stories, convert_to_tensor=True)

In [19]:
def semantic_search_1(input_data):
  title_list = []
  story_list = []
  score_list = []

  results = pd.DataFrame()
  # Find the closest 5 stories of the corpus for each query sentence based on cosine similarity
  top_k = min(6, len(story_embeddings))

  # If the input is too short to be a story or its not in the dataset
  if len(input_data) < 20 and input_data not in titles:
    print('Title Not Found')

  # If input is in the dataset
  elif input_data in titles:
    query_embeddings = embedder.encode(str(df_data[df_data['Title'] == input_data]['genres_keywords'])[5:-33], convert_to_tensor=True)

    # Use cosine-similarity and torch.topk to find the highest 5 scores
    cos_scores = util.pytorch_cos_sim(query_embeddings, story_embeddings)[0]
    top_results = torch.topk(cos_scores, k=65)

    input_data_2 = input_data.replace('.', '.\n')
    print("\n\n======================")
    print("    TOP RESULTS")
    print("======================\n")

    # For score, index in torch.topk(cos_scores, k=top_k) use index  locator for feature lists
    # push score to cpu and convert to 1D array
    for score, idx in zip(top_results[0], top_results[1]):
      title_list.append(titles[idx])
      story_list.append(stories[idx])
      score_list.append(score.cpu().numpy().flatten())

    results['Title'] = title_list
    results['Story'] = story_list
    results['Score'] = score_list
    # return dictionary
    return results.iloc[1:, :]

  # If the input is long enough to be a story which is in the dataset
  elif len(input_data) > 20 and input_data not in titles:
    # Find the closest 5 stories of the corpus for each query sentence based on cosine similarity
    query_embeddings = embedder.encode(input_data, convert_to_tensor=True)

    # Use cosine-similarity and torch.topk to find the highest 5 scores
    cos_scores = util.pytorch_cos_sim(query_embeddings, story_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)

    input_data = input_data.replace('.', '.\n')
    print(input_data)
    print("\n\n======================")
    print("    TOP RESULTS")
    print("======================\n")

    # For score, index in torch.topk(cos_scores, k=top_k) use index  locator for feature lists
    # push score to cpu and convert to 1D array
    for score, idx in zip(top_results[0], top_results[1]):
      title_list.append(titles[idx])
      story_list.append(stories[idx])
      score_list.append(score.cpu().numpy().flatten())

    # Push results to dictionary columns
    results['Title'] = title_list
    results['Story'] = story_list
    results['Score'] = score_list
    # return dictionary
    return results

In [26]:
semantic_search("뮤지컬 〈문스토리〉")



    TOP RESULTS



,Title,Story,Score
1,뮤지컬 〈렛미플라이〉,역사극 드라마 아폴로 호가 달을 향해 쏘아 올려진 년의 밤\n동네 최고의 수선장이...,[0.52778757]
2,뮤지컬 〈판〉,코믹 블랙코미디 감동 세기 말 조선 서민들 사이에서 흉흉한 세상을 풍자하는 패관...,[0.4148951]
3,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",역사극 판타지 시조가 국가이념인 상상 속의 조선\n삶의 고단함과 역경을 시조 속에...,[0.4006812]
4,뮤지컬 〈빨래〉,드라마 감동 공감/힐링 서울살이 년 차인 나영과 솔롱고 작가의 꿈을 안고 서울에...,[0.39863056]
5,뮤지컬 〈투모로우 모닝〉,로맨스 공감/힐링 통통 튀는 매력 존 캣의 풋풋함과\n책 캐서린의 무르익은 감...,[0.39050323]
...,...,...,...
60,뮤지컬 〈구텐버그〉,역사극 중세 독일의 작은 마을 슐리머\n포도즙을 짜던 평범한 남자 구텐버그가\n인쇄...,[0.1908222]
61,런던레코드,드라마 감동 런던 외곽에 위치한 낡고 오래된 레코드 샵 주인 존은 오늘도 하루를 ...,[0.16762534]
62,뮤지컬 프리즌,관객 참여형 코믹 락 밴드 연습생들 감옥 가다 가수의 꿈을 안고 혹독한 준비를 해...,[0.15406935]
63,난타(NANTA) - 명동공연,코믹 액션 세 명의 요리사가 하루 일과를 시작한다 즐겁게 요리를 준비하는 동안 지...,[0.1449534]


In [21]:
semantic_search_1("뮤지컬 〈문스토리〉")



    TOP RESULTS



,Title,Story,Score
1,뮤지컬 〈렛미플라이〉,역사극 드라마 아폴로 호가 달을 향해 쏘아 올려진 년의 밤\n동네 최고의 수선장이...,[0.52778757]
2,뮤지컬 〈판〉,코믹 블랙코미디 감동 세기 말 조선 서민들 사이에서 흉흉한 세상을 풍자하는 패관...,[0.4148951]
3,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",역사극 판타지 시조가 국가이념인 상상 속의 조선\n삶의 고단함과 역경을 시조 속에...,[0.4006812]
4,뮤지컬 〈빨래〉,드라마 감동 공감/힐링 서울살이 년 차인 나영과 솔롱고 작가의 꿈을 안고 서울에...,[0.39863056]
5,뮤지컬 〈투모로우 모닝〉,로맨스 공감/힐링 통통 튀는 매력 존 캣의 풋풋함과\n책 캐서린의 무르익은 감...,[0.39050323]
...,...,...,...
60,뮤지컬 〈구텐버그〉,역사극 중세 독일의 작은 마을 슐리머\n포도즙을 짜던 평범한 남자 구텐버그가\n인쇄...,[0.1908222]
61,런던레코드,드라마 감동 런던 외곽에 위치한 낡고 오래된 레코드 샵 주인 존은 오늘도 하루를 ...,[0.16762534]
62,뮤지컬 프리즌,관객 참여형 코믹 락 밴드 연습생들 감옥 가다 가수의 꿈을 안고 혹독한 준비를 해...,[0.15406935]
63,난타(NANTA) - 명동공연,코믹 액션 세 명의 요리사가 하루 일과를 시작한다 즐겁게 요리를 준비하는 동안 지...,[0.1449534]


In [28]:
a = semantic_search_1("뮤지컬 〈문스토리〉")
a



    TOP RESULTS



,Title,Story,Score
1,뮤지컬 〈렛미플라이〉,역사극 드라마 아폴로 호가 달을 향해 쏘아 올려진 년의 밤\n동네 최고의 수선장이...,[0.52778757]
2,뮤지컬 〈판〉,코믹 블랙코미디 감동 세기 말 조선 서민들 사이에서 흉흉한 세상을 풍자하는 패관...,[0.4148951]
3,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",역사극 판타지 시조가 국가이념인 상상 속의 조선\n삶의 고단함과 역경을 시조 속에...,[0.4006812]
4,뮤지컬 〈빨래〉,드라마 감동 공감/힐링 서울살이 년 차인 나영과 솔롱고 작가의 꿈을 안고 서울에...,[0.39863056]
5,뮤지컬 〈투모로우 모닝〉,로맨스 공감/힐링 통통 튀는 매력 존 캣의 풋풋함과\n책 캐서린의 무르익은 감...,[0.39050323]
...,...,...,...
60,뮤지컬 〈구텐버그〉,역사극 중세 독일의 작은 마을 슐리머\n포도즙을 짜던 평범한 남자 구텐버그가\n인쇄...,[0.1908222]
61,런던레코드,드라마 감동 런던 외곽에 위치한 낡고 오래된 레코드 샵 주인 존은 오늘도 하루를 ...,[0.16762534]
62,뮤지컬 프리즌,관객 참여형 코믹 락 밴드 연습생들 감옥 가다 가수의 꿈을 안고 혹독한 준비를 해...,[0.15406935]
63,난타(NANTA) - 명동공연,코믹 액션 세 명의 요리사가 하루 일과를 시작한다 즐겁게 요리를 준비하는 동안 지...,[0.1449534]


In [29]:
print(type(a))

<class 'pandas.core.frame.DataFrame'>


In [31]:
selected_columns = a[['Title', 'Score']]
selected_columns['Score'] = selected_columns['Score'].apply(lambda x: x[0])
selected_columns

<ipython-input-31-534d2ed530bf>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_columns['Score'] = selected_columns['Score'].apply(lambda x: x[0])


,Title,Score
1,뮤지컬 〈렛미플라이〉,0.527788
2,뮤지컬 〈판〉,0.414895
3,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",0.400681
4,뮤지컬 〈빨래〉,0.398631
5,뮤지컬 〈투모로우 모닝〉,0.390503
...,...,...
60,뮤지컬 〈구텐버그〉,0.190822
61,런던레코드,0.167625
62,뮤지컬 프리즌,0.154069
63,난타(NANTA) - 명동공연,0.144953


In [32]:
plot_genre_weight = selected_columns.set_index(keys=['Title'])
plot_genre_weight

,Score
Title,
뮤지컬 〈렛미플라이〉,0.527788
뮤지컬 〈판〉,0.414895
"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",0.400681
뮤지컬 〈빨래〉,0.398631
뮤지컬 〈투모로우 모닝〉,0.390503
...,...
뮤지컬 〈구텐버그〉,0.190822
런던레코드,0.167625
뮤지컬 프리즌,0.154069


# 별점 가중치 함수


*   변수가 적절히 잘 쓰였는지 확인(코드 이해)
*   변경할 점 있으면 변수 바꾼다.



연극 별점가중치

In [33]:
# 연극_평점
import pandas as pd
play_star = pd.read_excel("../data/raw/playTop77_star.xlsx")
play_star

,Unnamed: 0,Title,ID,Star,Date
0,5316,연극 〈운빨로맨스〉- 대학로,seeun0***,5,2023.10.10
1,5317,연극 〈운빨로맨스〉- 대학로,isoo9***,5,2023.10.10
2,5318,연극 〈운빨로맨스〉- 대학로,rkgml5***,5,2023.10.09
3,5324,연극 〈운빨로맨스〉- 대학로,da9***,5,2023.10.03
4,5325,연극 〈운빨로맨스〉- 대학로,4322***,5,2023.09.28
...,...,...,...,...,...
9725,9818,그곳에 있었다,judy7***,5,2023.09.14
9726,9819,그곳에 있었다,ddhj***,5,2023.09.03
9727,9820,그곳에 있었다,gustnr4***,5,2023.09.03
9728,9821,그곳에 있었다,pinkhwan***,5,2023.09.03


In [34]:
print(type(play_star))

<class 'pandas.core.frame.DataFrame'>


In [35]:
star = play_star['Star']
print(play_star.shape[0])
num_play = play_star.shape[0]
print(type(num_play))

9730
<class 'int'>



코드 설명
1.   1+review_count: 리뷰 수에 1을 더함. 분모가 0이되는 것을 방지하기 위해.
2.   항목 추가



In [36]:
# user_ratings : 별점리스트, review_count: 전체 행개수
def calculate_weighted_rating (user_ratings, review_count, base_weight=1.0):

  # 가중치 계산
  weight = base_weight*( 1.0 + review_count)    # 사람 수가 적을수록 가중치 감소/ 기본 가중치값 먼저 필요

  # 평균 별점 계산
  average_rating = sum(user_ratings)/len(user_ratings)   # 작품명 같은 것끼리 계산

  # 가중 평균 별점 계산
  weighted_rating = average_rating*weight

  return weighted_rating

## 데이터 함수 적용

*   같은 작품끼리 어떻게 묶을 것인지 정해야함
*   for문 이용해서 작품마다 함수 돌려야함



In [37]:
# 일단 작품이름 같은 것끼리 묶어야함
# 묶은 것에서 user_rating 리스트 만들기
# 묶은 것 아이템 개수=작품별 리뷰 개수 계산
# for문 이용해서 작품 하나씩 들어가서 돌려야함.
# 결과  --> 작품: 가중치 이런식으로 아웃풋.

In [38]:
# 제목별로 별점 그룹화
grouped_title = play_star.groupby('Title')['Star'].apply(list).reset_index()
type(grouped_title)

pandas.core.frame.DataFrame

In [39]:
# 별점 가중치 함수 데이터에 적용

review_count = 100
weighted_ratings = {}

for row in grouped_title.itertuples():
  title = row.Title
  user_ratings = row.Star

  # 가중 평균 계산 함수 호출
  weighted_rating = calculate_weighted_rating(user_ratings, review_count=len(user_ratings))
  weighted_ratings[title] = weighted_rating


for title, weighted_rating in weighted_ratings.items():
    print(f"Title: {title}, Weighted Rating: {weighted_rating}")

Title: (리얼타임 코믹연극) 택시안에서 - 서울, Weighted Rating: 1005.744075829384
Title: (코믹연극) 달동네-부산, Weighted Rating: 73.92857142857143
Title: 10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀, Weighted Rating: 2462.819607843137
Title: 2023 연극 〈친정엄마와 2박 3일〉 - 고양, Weighted Rating: 33.142857142857146
Title: 2시간탈출 졸탄쇼, Weighted Rating: 653.7720588235295
Title: 3대가 웃고 우는 연극 〈염쟁이 유씨〉, Weighted Rating: 68.92307692307693
Title: 4D공포연극 〈스위치〉, Weighted Rating: 2457.9555555555557
Title: ★평점9.5★ 코미디의맛/쇼미더퍼니, Weighted Rating: 2124.7640449438204
Title: 공포스릴러연극〈두여자〉- 대구, Weighted Rating: 310.0
Title: 공포연극 조각, Weighted Rating: 617.7890625
Title: 공포연극 ［자취］, Weighted Rating: 966.81
Title: 국민 코믹 연극 〈오백에삼십〉 - 대학로 세우아트센터 1관, Weighted Rating: 2800.8966725043783
Title: 그곳에 있었다, Weighted Rating: 57.81818181818182
Title: 나의 장례식에 와줘, Weighted Rating: 73.92857142857143
Title: 대학로 1위 연극 〈쉬어매드니스〉, Weighted Rating: 2432.7607843137253
Title: 대학로 청소년연극 〈사춘기메들리〉, Weighted Rating: 716.9444444444445
Title: 로맨틱코미디 〈슬기로운 신혼생활〉, Weighted Rating: 533.89814814

In [40]:
def star_weight(excel):
  import pandas as pd
  star_df = pd.read_excel(excel)
  star = star_df['Star']
  num_play = star_df.shape[0]

  # 일단 작품이름 같은 것끼리 묶어야함
  # 묶은 것에서 user_rating 리스트 만들기
  # 묶은 것 아이템 개수=작품별 리뷰 개수 계산
  # for문 이용해서 작품 하나씩 들어가서 돌려야함.
  # 결과  --> 작품: 가중치 이런식으로 아웃풋.

  # 제목별로 별점 그룹화
  grouped_title = star_df.groupby('Title')['Star'].apply(list).reset_index()
  type(grouped_title)

  # 별점 가중치 함수 데이터에 적용

  review_count = 100
  weighted_ratings = {}

  for row in grouped_title.itertuples():
    title = row.Title
    user_ratings = row.Star

    # 가중 평균 계산 함수 호출
    weighted_rating = calculate_weighted_rating(user_ratings, review_count=len(user_ratings))
    weighted_ratings[title] = weighted_rating


  for title, weighted_rating in weighted_ratings.items():
      print(f"Title: {title}, Weighted Rating: {weighted_rating}")
  return weighted_ratings

In [41]:
play_weighted_ratings = star_weight("../data/raw/playTop77_star.xlsx")
musical_weighted_ratings = star_weight("../data/raw/musicalTop77_star.xlsx")

Title: (리얼타임 코믹연극) 택시안에서 - 서울, Weighted Rating: 1005.744075829384
Title: (코믹연극) 달동네-부산, Weighted Rating: 73.92857142857143
Title: 10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀, Weighted Rating: 2462.819607843137
Title: 2023 연극 〈친정엄마와 2박 3일〉 - 고양, Weighted Rating: 33.142857142857146
Title: 2시간탈출 졸탄쇼, Weighted Rating: 653.7720588235295
Title: 3대가 웃고 우는 연극 〈염쟁이 유씨〉, Weighted Rating: 68.92307692307693
Title: 4D공포연극 〈스위치〉, Weighted Rating: 2457.9555555555557
Title: ★평점9.5★ 코미디의맛/쇼미더퍼니, Weighted Rating: 2124.7640449438204
Title: 공포스릴러연극〈두여자〉- 대구, Weighted Rating: 310.0
Title: 공포연극 조각, Weighted Rating: 617.7890625
Title: 공포연극 ［자취］, Weighted Rating: 966.81
Title: 국민 코믹 연극 〈오백에삼십〉 - 대학로 세우아트센터 1관, Weighted Rating: 2800.8966725043783
Title: 그곳에 있었다, Weighted Rating: 57.81818181818182
Title: 나의 장례식에 와줘, Weighted Rating: 73.92857142857143
Title: 대학로 1위 연극 〈쉬어매드니스〉, Weighted Rating: 2432.7607843137253
Title: 대학로 청소년연극 〈사춘기메들리〉, Weighted Rating: 716.9444444444445
Title: 로맨틱코미디 〈슬기로운 신혼생활〉, Weighted Rating: 533.89814814

# 배우 가중치 함수

## PlayDB 누적조회순 랭킹 불러오기

---

 - actor_dic 딕셔너리 구조

{ 배우 : { 'index': 순위, 'masterpiece' : [최근공연 3개 리스트] } }




In [42]:
import pandas as pd
def actor_lanking(csv):
  actor_csv = pd.read_csv(csv, encoding='utf-8')
  actor_csv.columns = ["index", "actor_name", "masterpiece"]

  # 랭킹에 있는 배우들의 작품만 따로 추출 (작품명이 \n으로 이어진 하나의 문자열이기 때문에, 리스트화 시킬 예정)
  actor_mp = actor_csv['masterpiece'].to_list()

  actor_dic = actor_csv.set_index('index').T.to_dict()  # 딕셔너리 형태로 변환

  # key: actor_name, value: index, masterpiece
  for key, value in actor_dic.items():
    temp = value['masterpiece']
    temp2 = temp.split('\n')
    value['masterpiece'] = temp2
  return actor_dic

In [43]:
play_actor_dic = actor_lanking("../data/rawtheater_actor_rank.csv")
play_actor_dic

{0: {'actor_name': '최성희', 'masterpiece': ['몽실(2009)']},
 1: {'actor_name': '김다현',
  'masterpiece': ['루쓰(2023)',
   '더 그레이티스트 뮤지컬 콘서트 - 용인(2022)',
   '미세스 다웃파이어(2022)']},
 2: {'actor_name': '김지현',
  'masterpiece': ['그날들 - 대구(2023)', '그날들(2023)', '스위니토드(2022)']},
 3: {'actor_name': '박호산',
  'masterpiece': ['기형도 플레이(2023)', '오셀로(2023)', '무제의 시대(2022)']},
 4: {'actor_name': '윤주상',
  'masterpiece': ['타클라마칸(2018)',
   '웃어요 덕구씨(2016)',
   '2012 세계국립극장페스티벌 - 인물실록 봉달수(2012)']},
 5: {'actor_name': '조재현',
  'masterpiece': ['블랙버드(2016)',
   '2016 여우락페스티벌 - 2 조재현 황석정 두번째달(2016)',
   '에쿠우스(2015)']},
 6: {'actor_name': '이현수',
  'masterpiece': ['제11회 대구국제뮤지컬페스티벌 - 아름다운 슬픈 날(2017)',
   '화려한 싱글들(2012)',
   '순이야 사랑해(2012)']},
 7: {'actor_name': '송영창',
  'masterpiece': ['브로드웨이 42번가(2018)', '모래시계(2017)', '올드위키드송(2016)']},
 8: {'actor_name': '송승환',
  'masterpiece': ['더 드레서(2021)', '더 드레서(2020)', '뮤 하트(2019)']},
 9: {'actor_name': '김성녀',
  'masterpiece': ['심청이와 춘향이가 온다－의정부(2023)',
   '죽음의 배 & 갈매기 - 수원(2023)'

In [44]:
mus_actor_dic = actor_lanking("../data/raw/musical_actor_rank.csv")
mus_actor_dic

{0: {'actor_name': '임태경',
  'masterpiece': ['할란카운티(2023)', '지저스 크라이스트 수퍼스타(2022)', '나폴레옹 헌정 콘서트(2022)']},
 1: {'actor_name': '김다현',
  'masterpiece': ['루쓰(2023)',
   '더 그레이티스트 뮤지컬 콘서트 - 용인(2022)',
   '미세스 다웃파이어(2022)']},
 2: {'actor_name': '차지연',
  'masterpiece': ['컴프롬어웨이(2023)', '차지연 콘서트(2023)', '아마데우스(2023)']},
 3: {'actor_name': '김지현',
  'masterpiece': ['그날들 - 대구(2023)', '그날들(2023)', '스위니토드(2022)']},
 4: {'actor_name': '바다',
  'masterpiece': ['안양 시 승격 50주년 기념 신년음악회 - 안양(2023)',
   '제주신화월드 카운트다운 2023 - 제주(2022)',
   '드라뮤지션 바다 콘서트 - 하남(2022)']},
 5: {'actor_name': '규현',
  'masterpiece': ['규현 & HYNN 콘서트(2023)', '벤허(2023)', '2023 서울파크뮤직페스티벌(2023)']},
 6: {'actor_name': '박호산',
  'masterpiece': ['기형도 플레이(2023)', '오셀로(2023)', '무제의 시대(2022)']},
 7: {'actor_name': '주지훈',
  'masterpiece': ['주지훈 팬미팅(2019)', '생명의 항해(2010)', '돈 주앙(2009)']},
 8: {'actor_name': '강태을',
  'masterpiece': ['몬테크리스토(2023)', '엘리자벳(2022)', '엑스칼리버(2022)']},
 9: {'actor_name': '정민',
  'masterpiece': ['스모크(2023)', '사의찬미 콘서트(2

## 인터파크 캐스팅 배우 불러오기


---

- inter_dic 딕셔너리 구조

{ 작품 제목 : { 배우 : [캐스팅 배우 리스트] } }

In [45]:
def cast_actor(excel):
  inter_actor = pd.read_excel(excel)
  inter_actor = inter_actor[['Title', 'cast_act']]

  # 문자열 형태로 되어 있는 cast_act 열을 리스트화 시켜서 다시 집어 넣을 예정
  inter_actor_cast = inter_actor['cast_act']

  inter_list = []
  for i in range(len(inter_actor)):
    string = inter_actor_cast[i]
    ch_string = string.split(', ')
    inter_list.append(ch_string)

  # Title 기준 딕셔너리화
  inter_dic = inter_actor.set_index('Title').T.to_dict()

  # key: Title, cast_act
  i = 0
  for key, value in inter_dic.items():
    value['cast_act'] = inter_list[i]
    i+=1
  return inter_dic, inter_actor

In [46]:
play_inter_dic, play_inter_actor = cast_actor("../data/raw/playTop77_casting.xlsx")
play_inter_dic

{'연극 〈운빨로맨스〉- 대학로': {'cast_act': ['김세울',
   '문재웅',
   '변가온',
   '한선우',
   '김지후',
   '장은서',
   '조은진',
   '최지선',
   '박병훈',
   '신광희',
   '이주영',
   '신서진',
   '민소현',
   '양혜민',
   '최로아']},
 '(리얼타임 코믹연극) 택시안에서 - 서울': {'cast_act': ['김로언',
   '전청일',
   '한상웅',
   '김나온',
   '한재우',
   '오하성',
   '이채연',
   '김진화',
   '서예선']},
 '(코믹연극) 달동네-부산': {'cast_act': ['조영준',
   '안애린',
   '한서아',
   '류현주',
   '양채은',
   '정유나',
   '곽희규',
   '김영신',
   '신희진',
   '이찬영',
   '이민진',
   '임유림',
   '오현석',
   '양준재',
   '정창운']},
 '10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀': {'cast_act': ['임채민',
   '정지호',
   '문지원',
   '정승지',
   '양시환',
   '황민환',
   '최지환',
   '장진호',
   '박지민',
   '심채아',
   '이주하',
   '정한슬',
   '김동진',
   '함원태',
   '김건하',
   '이선준']},
 '2023 연극 〈친정엄마와 2박 3일〉 - 고양': {'cast_act': ['해당없음']},
 '2시간탈출 졸탄쇼': {'cast_act': ['해당없음']},
 '4D공포연극 〈스위치〉': {'cast_act': ['김상훈',
   '엄민욱',
   '전재진',
   '이창주',
   '최하은',
   '정재희',
   '최유경',
   '김시원',
   '김정규',
   '나준영',
   '함우주',
   '김창우',
   '윤수아',
   '김단하',
   '박애림']},
 '★평점9.5★ 코미디의맛/쇼미더퍼니': {'cast

In [47]:
play_inter_actor

,Title,cast_act
0,연극 〈운빨로맨스〉- 대학로,"김세울, 문재웅, 변가온, 한선우, 김지후, 장은서, 조은진, 최지선, 박병훈, 신..."
1,(리얼타임 코믹연극) 택시안에서 - 서울,"김로언, 전청일, 한상웅, 김나온, 한재우, 오하성, 이채연, 김진화, 서예선"
2,(코믹연극) 달동네-부산,"조영준, 안애린, 한서아, 류현주, 양채은, 정유나, 곽희규, 김영신, 신희진, 이..."
3,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,"임채민, 정지호, 문지원, 정승지, 양시환, 황민환, 최지환, 장진호, 박지민, 심..."
4,2023 연극 〈친정엄마와 2박 3일〉 - 고양,해당없음
...,...,...
60,공포스릴러연극〈두여자〉- 대구,"김미영, 김린아, 이다은, 엄영실, 이상아, 임성희, 이도윤, 최승환, 문다현, 박..."
61,모든 날 모든 순간,해당없음
62,나의 장례식에 와줘,해당없음
63,3대가 웃고 우는 연극 〈염쟁이 유씨〉,해당없음


In [48]:
mus_inter_dic, mus_inter_actor = cast_actor("/content/drive/MyDrive/캡스톤/데이터/뮤지컬/musicalTop77_casting.xlsx")
mus_inter_dic

{'2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천': {'cast_act': ['해당없음']},
 '2023 창작뮤지컬어워드 NEXT': {'cast_act': ['해당없음']},
 '2023 최현우 Answer - 성남': {'cast_act': ['최현우']},
 '2023~24 최현우 Answer': {'cast_act': ['최현우']},
 '2023송승환의 오리지널 난타(대전)': {'cast_act': ['해당없음']},
 '2024 최현우 Answer - 청주': {'cast_act': ['최현우']},
 '2023 매직쇼〈최현우의 미스틱커스〉 - 평택': {'cast_act': ['해당없음']},
 '국립창극단 〈패왕별희〉': {'cast_act': ['해당없음']},
 '난타(NANTA) - 명동공연': {'cast_act': ['해당없음']},
 '넌버벌 코미디 〈옹알스〉': {'cast_act': ['조수원',
   '조준우',
   '채경선',
   '최기섭',
   '하박',
   '이경섭',
   '최진영']},
 '더 맨 얼라이브 〈초이스〉': {'cast_act': ['이유청', '강청광', '김준석', '김성길', '박휘범', '강태욱']},
 '동구문화체육센터 뮤지컬 〈앤 ANNE〉 - 인천': {'cast_act': ['천우주']},
 '라면에 파송송': {'cast_act': ['김재선', '김슬기', '김태경']},
 '러스트(RUST) - 전주': {'cast_act': ['해당없음']},
 '런던레코드': {'cast_act': ['이채원', '해리 벤자민']},
 '뮤지컬 〈22년 2개월〉': {'cast_act': ['유승현',
   '양지원',
   '이재환',
   '최수진',
   '강혜인',
   '홍나현',
   '유성재',
   '안창용',
   '정호준',
   '이현재',
   '박세훈',
   '성재',
   '정종환',
   '박상선',
   '신요셉']},
 '뮤지컬 〈곤 투모로우〉': {'c

In [49]:
mus_inter_actor

,Title,cast_act
0,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",해당없음
1,2023 창작뮤지컬어워드 NEXT,해당없음
2,2023 최현우 Answer - 성남,최현우
3,2023~24 최현우 Answer,최현우
4,2023송승환의 오리지널 난타(대전),해당없음
...,...,...
60,뮤지컬 썸데이,"김여진, 서찬양, 김아름, 이채림, 구문회, 이제성, 김선오, 윤동환, 박소진, 조..."
61,뮤지컬 프리즌,"손동진, 최승환, 홍차민, 안중현, 김세훈, 신연우, 하준석, 이강오, 김동훈, 추..."
62,뮤직드라마 당신만이,"장인혁, 박민승, 박준성, 이일진, 손예슬, 이민지, 박세웅, 차은진, 권성민, 이경수"
63,뮤지컬 〈투모로우 모닝〉,"렉스, 정현준, 천관우, 김안젤라, 박혜원, 한수민, 문준혁, 임창민, 오태후, 이..."


## 배우명 비교
* actor_dic : playDB 랭킹별 최근공연 딕셔너리
* inter_dic : 인터파크 작품별 캐스팅 배우 딕셔너리


---

> 찾는 배우 이름(인터파크 캐스팅 배우)이 actor_dic에 있을 경우, 해당 배우의 랭킹 도출

→ 해당 배우의 랭킹(인덱스)를 활용하여 최근 공연 리스트 불러오기

→ 최근 공연 리스트와 현재 찾는 배우의 공연 이름 비교

* 문제: 배우 최근 공연과 인터파크 제목이 다르므로 배우의 최근 공연을 " - "으로 split 한 후, 인터파크 제목.find(최근 공연) 하는 함수를 짜면 될 듯

In [50]:
# 프로토타입 (이건 playDB 동명이인이 있을 경우)
res = []
for key, value in play_actor_dic.items():
  if ('최성희'.find(value['actor_name']) > -1):
    res.append(key)
res

[0]

In [51]:
# 배우 이름 찾기 (랭킹 도출) 함수
def find_value(dictionary, finding_value):
    for key, value in dictionary.items():
        if (finding_value.find(value['actor_name']) > -1):
            return key


In [52]:
def search_actor(inter_dic, actor_dic):
  actor_mp = {}
  i = 0
  for key1, value1 in inter_dic.items():
      for name in value1['cast_act']:
          index = find_value(actor_dic, name)
          if index is not None:
              for key2, value2 in actor_dic.items():
                  if key2 == index:
                    # 여기서부터 짰음
                      recent_performances = value2['masterpiece']

                      for title in recent_performances:
                        title = title.split('(')
                        title = title[0]

                        # 인터파크 제목에서 최근 공연을 찾음
                        temp = key1.find(title)
                        if temp > -1:
                          print(f"배우 {name}의 최근 공연 '{key1}'이 인터파크 제목에 포함됨.")
                          actor_mp[i] = {'actor' : name, 'Title' : key1}
                          i+=1
  return actor_mp

### 밑 반복문 구조 설명

* for key1, value1 in inter_dic.items(): 인터파크 캐스팅 배우 딕셔너리를 key, value로 쪼갬

* for name in value1['cast_act']: 작품별 캐스팅배우의 이름 기준으로 반복

* index = find_value(actor_dic, name) : 배우 이름 찾기 함수로 배우 찾아서 랭킹을 인덱스에 저장

* if (index!=None): 인덱스가 None이면 랭킹에 없다는 뜻

* for key2, value2 in actor_dic.items(): 배우 랭킹 딕셔너리를 key, value로 쪼갬
        
* if (value2['index'] == index): 랭킹(인덱스)이 같다면

* print(value2['masterpiece']) : 작품명 출력 (여기부터 바꾸면 될 듯)



---


* 연극 배우

In [53]:
PSA = search_actor(play_inter_dic, play_actor_dic)

배우 박은영의 최근 공연 '뮤직드라마 〈불편한 편의점〉'이 인터파크 제목에 포함됨.
배우 최덕문의 최근 공연 '연극 〈기형도 플레이〉'이 인터파크 제목에 포함됨.
배우 박호산의 최근 공연 '연극 〈기형도 플레이〉'이 인터파크 제목에 포함됨.


In [54]:
MSA = search_actor(mus_inter_dic, mus_actor_dic)

배우 이재환의 최근 공연 '뮤지컬 〈22년 2개월〉'이 인터파크 제목에 포함됨.
배우 정민의 최근 공연 '뮤지컬 〈구텐버그〉'이 인터파크 제목에 포함됨.
배우 김지현의 최근 공연 '뮤지컬 〈그날들〉 10주년 기념 공연 - 대전'이 인터파크 제목에 포함됨.
배우 린아의 최근 공연 '뮤지컬 〈레미제라블〉 - 부산'이 인터파크 제목에 포함됨.
배우 정민의 최근 공연 '뮤지컬 〈스모크〉'이 인터파크 제목에 포함됨.
배우 김신의의 최근 공연 '뮤지컬 〈삼총사〉'이 인터파크 제목에 포함됨.
배우 김신의의 최근 공연 '뮤지컬 〈삼총사〉'이 인터파크 제목에 포함됨.
배우 송원근의 최근 공연 '뮤지컬 〈오페라의 유령〉 - 서울'이 인터파크 제목에 포함됨.
배우 김아선의 최근 공연 '뮤지컬 〈오페라의 유령〉 - 서울'이 인터파크 제목에 포함됨.
배우 차지연의 최근 공연 '뮤지컬 〈컴프롬어웨이〉'이 인터파크 제목에 포함됨.
배우 문혜원의 최근 공연 '뮤지컬 〈타오르는 어둠 속에서〉'이 인터파크 제목에 포함됨.
배우 송상은의 최근 공연 '뮤지컬 〈블랙메리포핀스〉'이 인터파크 제목에 포함됨.
배우 규현의 최근 공연 '뮤지컬 ＇벤허＇'이 인터파크 제목에 포함됨.
배우 이희정의 최근 공연 '뮤지컬 ＇벤허＇'이 인터파크 제목에 포함됨.
배우 이자람의 최근 공연 '창작가무극 〈순신〉'이 인터파크 제목에 포함됨.
배우 고미경의 최근 공연 '창작가무극 〈순신〉'이 인터파크 제목에 포함됨.


In [55]:
for key1, value1 in play_inter_dic.items():
    for name in value1['cast_act']:
        index = find_value(play_actor_dic, name)
        if index is not None:
            for key2, value2 in play_actor_dic.items():
                if key2 == index:
                  # 여기서부터 짰음
                    recent_performances = value2['masterpiece']

                    for title in recent_performances:
                      title = title.split('(')
                      title = title[0]

                      # 인터파크 제목에서 최근 공연을 찾음
                      temp = key1.find(title)
                      if temp > -1:
                        print(f"배우 {name}의 최근 공연 '{key1}'이 인터파크 제목에 포함됨.")

    #print(name + " : " + str(index))

배우 박은영의 최근 공연 '뮤직드라마 〈불편한 편의점〉'이 인터파크 제목에 포함됨.
배우 최덕문의 최근 공연 '연극 〈기형도 플레이〉'이 인터파크 제목에 포함됨.
배우 박호산의 최근 공연 '연극 〈기형도 플레이〉'이 인터파크 제목에 포함됨.




---

* 뮤지컬 배우

In [56]:
for key1, value1 in mus_inter_dic.items():
    for name in value1['cast_act']:
        index = find_value(mus_actor_dic, name)
        if index is not None:
            for key2, value2 in mus_actor_dic.items():
                if key2 == index:
                  # 여기서부터 짰음
                    recent_performances = value2['masterpiece']

                    for title in recent_performances:
                      title = title.split('(')
                      title = title[0]

                      # 인터파크 제목에서 최근 공연을 찾음
                      temp = key1.find(title)
                      if temp > -1:
                        print(f"배우 {name}의 최근 공연 '{key1}'이 인터파크 제목에 포함됨.")

    #print(name + " : " + str(index))

배우 이재환의 최근 공연 '뮤지컬 〈22년 2개월〉'이 인터파크 제목에 포함됨.
배우 정민의 최근 공연 '뮤지컬 〈구텐버그〉'이 인터파크 제목에 포함됨.
배우 김지현의 최근 공연 '뮤지컬 〈그날들〉 10주년 기념 공연 - 대전'이 인터파크 제목에 포함됨.
배우 린아의 최근 공연 '뮤지컬 〈레미제라블〉 - 부산'이 인터파크 제목에 포함됨.
배우 정민의 최근 공연 '뮤지컬 〈스모크〉'이 인터파크 제목에 포함됨.
배우 김신의의 최근 공연 '뮤지컬 〈삼총사〉'이 인터파크 제목에 포함됨.
배우 김신의의 최근 공연 '뮤지컬 〈삼총사〉'이 인터파크 제목에 포함됨.
배우 송원근의 최근 공연 '뮤지컬 〈오페라의 유령〉 - 서울'이 인터파크 제목에 포함됨.
배우 김아선의 최근 공연 '뮤지컬 〈오페라의 유령〉 - 서울'이 인터파크 제목에 포함됨.
배우 차지연의 최근 공연 '뮤지컬 〈컴프롬어웨이〉'이 인터파크 제목에 포함됨.
배우 문혜원의 최근 공연 '뮤지컬 〈타오르는 어둠 속에서〉'이 인터파크 제목에 포함됨.
배우 송상은의 최근 공연 '뮤지컬 〈블랙메리포핀스〉'이 인터파크 제목에 포함됨.
배우 규현의 최근 공연 '뮤지컬 ＇벤허＇'이 인터파크 제목에 포함됨.
배우 이희정의 최근 공연 '뮤지컬 ＇벤허＇'이 인터파크 제목에 포함됨.
배우 이자람의 최근 공연 '창작가무극 〈순신〉'이 인터파크 제목에 포함됨.
배우 고미경의 최근 공연 '창작가무극 〈순신〉'이 인터파크 제목에 포함됨.


## 배우 가중치

* 연극 배우 가중치

* 동명이인 제거 전 : 702명

* 현재는 동명이인의 경우 이름1, 이름2 와 같이 개별적으로 구분
* 만약 이게 필요없다면, 원래 원본 파일을 가지고 실행하면 됨

In [57]:
def actor_name_dic(actor_dic, inter_dic, SA):
  dic = {}
  for key1, value1 in actor_dic.items():
    dic[key1] = value1['actor_name']


  index = 100  # 순위권 이후 배우 추가용
  for key1, value1 in inter_dic.items():
    for i in value1['cast_act']:  # 공연별 캐스팅 배우 딕셔너리화
      ret = 0 # 순위권 배우 이름 중복 추가 방지용
      for key2, value2 in SA.items(): # 순위권 배우 판별
        if (value2['Title'] == key1 and value2['actor'] == i) or (i == '해당없음'): # 순위권 배우거나, 배우명이 없는 경우 추가 제외
          ret = 1
          break
      if(ret == 0):
        for ran in range(100, index):  # 동명이인 찾기
          if(dic[ran] == i ):
            ret = 1
            break
          else:
            ret = 0

      if(ret == 0): # 그 외 전부 추가
        dic[index] = i
        index += 1
  dic_t = {v:k for k,v in dic.items()}
  return dic, dic_t

In [58]:
pt_actor_dic, pt_actor_dic_t = actor_name_dic(play_actor_dic, play_inter_dic, PSA)

* 랭킹 배우의 최근 공연작 중 데이터에 해당하는 것이 있다면, 배우 이름 -> 인덱스

In [59]:
def change_index(search, dic):
  ret_dic = {}
  for key, value in search.items():
    index = dic[value['actor']]
    ret_dic[index] = value['Title']
  return ret_dic

* 모든 배우의 이름을 인덱스로 교체 + 공연별 배우 리스트 생성

In [60]:
def cast_list(dic1, dic2, dic_t):
  cast_dic = {}
  for key1, value1 in dic2.items():
    cast_act = value1['cast_act']
    cast_index = []
    # 배우 인덱스 저장
    for name in cast_act:
      cast_index.append(dic_t.get(name))
    cast_dic[key1] = cast_index
  return cast_dic

* 배우의 가중치 구하는 함수
  * 사용자가 입력한 공연의 배우가 다른 공연에 있을 경우 +1점
  * 유명한 배우의 공연은 무조건 +2.5점

* 나중에 별점 등의 가중치들과 더하거나 곱할 예정

In [61]:
def actor_rate(cast_dic, title_name, ci):
  score_list = []
  cast_list = cast_dic[title_name]
  for key, value in cast_dic.items():
    score = 0
    for index in cast_list:
      # 이 공연의 배우가 있을 경우 추천점수 +1점
      if((key != title_name) and (index in value)):
        score += 1
      # 유명한 배우가 있을 경우 추천점수 +2.5점
    for name, title in ci.items():
      if(key == title):
        score += 2.5
    score_list.append(score)
  return score_list

In [62]:
play_ci = change_index(PSA, pt_actor_dic_t) # 랭킹배우의 이름->인덱스
play_ci

{51: '뮤직드라마 〈불편한 편의점〉', 21: '연극 〈기형도 플레이〉', 3: '연극 〈기형도 플레이〉'}

In [63]:
play_cast_list = cast_list(pt_actor_dic, play_inter_dic, pt_actor_dic_t)
score_list = actor_rate(play_cast_list, '진짜나쁜소녀', play_ci)
play_inter_actor['actor_score'] = score_list
play_inter_actor.sort_values(by=['actor_score'], ascending=False)

,Title,cast_act,actor_score
18,연극 〈기형도 플레이〉,"최덕문, 우현주, 이석준, 박호산, 이창훈, 이동하, 이은, 김세영, 김승은",5.0
15,뮤직드라마 〈불편한 편의점〉,"장우진, 김동박, 이강혁, 박은영, 조영임, 문상희, 정예지, 정이수, 서지영, 한...",2.5
37,행오버,"최선량, 정현규, 조영광, 신재명, 손준표, 윤건웅, 최창빈, 김민우, 이도희, 이...",1.0
59,로맨틱코미디 〈슬기로운 신혼생활〉,"오세영, 정대경, 김은지, 이슬기, 한재우, 서동현, 김세나, 이재희, 송민주, 이...",1.0
28,연극 한뼘사이,"윤주희, 강단비, 주하영, 유이주, 박정윤, 최하은, 김시윤, 김현지, 김종훈, 정...",1.0
...,...,...,...
27,연극 죽여주는 이야기,"남경화, 노진욱, 이연승, 최다운, 조웅희",0.0
29,연극 핫식스,"이대우, 정회형, 김호준, 김예지, 송다영, 김은수, 임종철, 김동섭, 김정원, 전상혁",0.0
30,연극〈늘근도둑이야기〉,"노진원, 전재형, 박철민, 태항호, 김상묵, 주우성, 유일한, 이호연, 안태영",0.0
31,이머시브씨어터 카지노,"안지현, 남지현, 문수민, 오자송, 최지우, 김유리, 박상혁, 장영웅, 박서연, 안...",0.0




---


* 뮤지컬 배우 가중치

In [64]:
mt_actor_dic, mt_actor_dic_t = actor_name_dic(mus_actor_dic, mus_inter_dic, MSA)

In [65]:
mus_ci = change_index(MSA, mt_actor_dic_t) # 랭킹배우의 이름->인덱스
mus_ci

{36: '뮤지컬 〈22년 2개월〉',
 9: '뮤지컬 〈스모크〉',
 50: '뮤지컬 〈그날들〉 10주년 기념 공연 - 대전',
 20: '뮤지컬 〈레미제라블〉 - 부산',
 47: '뮤지컬 〈삼총사〉',
 13: '뮤지컬 〈오페라의 유령〉 - 서울',
 29: '뮤지컬 〈오페라의 유령〉 - 서울',
 2: '뮤지컬 〈컴프롬어웨이〉',
 15: '뮤지컬 〈타오르는 어둠 속에서〉',
 32: '뮤지컬 〈블랙메리포핀스〉',
 5: '뮤지컬 ＇벤허＇',
 25: '뮤지컬 ＇벤허＇',
 11: '창작가무극 〈순신〉',
 99: '창작가무극 〈순신〉'}

In [66]:
mus_cast_list = cast_list(mt_actor_dic, mus_inter_dic, mt_actor_dic_t)
score_list = actor_rate(mus_cast_list, '뮤지컬 〈오페라의 유령〉 - 서울', mus_ci)
mus_inter_actor['actor_score'] = score_list
mus_inter_actor.sort_values(by=['actor_score'], ascending=False)

,Title,cast_act,actor_score
38,뮤지컬 〈오페라의 유령〉 - 서울,"조승우, 최재림, 김주택, 전동석, 손지수, 송은혜, 송원근, 황건하, 윤영석, 이...",5.0
51,뮤지컬 ＇벤허＇,"박은태, 신성록, 규현, 이지훈, 박민성, 서경수, 윤공주, 이정화, 최지혜, 이정...",5.0
54,창작가무극 〈순신〉,"형남희, 최인형, 이자람, 윤제원, 권성찬, 송문선, 고미경, 금승훈, 김백현, 김...",5.0
21,뮤지컬 〈레미제라블〉 - 부산,"민우혁, 최재림, 김우형, 카이, 조정은, 린아, 임기홍, 육현욱, 박준면, 김영주...",3.5
35,뮤지컬 〈스모크〉,"김재범, 정민, 김경수, 박정원, 강찬, 손유동, 홍승안, 김지유, 김청아, 장보람...",2.5
...,...,...,...
28,뮤지컬 〈문스토리〉,"정상윤, 성연, 김진욱, 김준호, 소정화, 박새힘, 주다온, 장보람, 강찬, 김리현...",0.0
29,뮤지컬 〈빨래〉,해당없음,0.0
30,쇼 뮤지컬 〈시스터즈 (SheStars!)〉,"유연, 신의정, 선민, 황성현, 김려원, 하유진, 정유지, 이예은, 이서영, 정연,...",0.0
31,뮤지컬 〈비더슈탄트〉,"송유택, 안지환, 황순종, 정백선, 김바다, 김지온, 동현, 이진우, 김이담, 손지...",0.0


# 가중치 총합 함수

In [67]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler((0,5))
def weight_calc(weighted_ratings):
  weighted_df = pd.DataFrame(weighted_ratings, index = ['StarScore']).T
  weighted_df[:] = scaler.fit_transform(weighted_df[:])
  return weighted_df

In [68]:
def castscore_calc(title, actor_dic, inter_dic, actor_dic_t, ci, inter_actor):
  casting_list = cast_list(actor_dic, inter_dic, actor_dic_t)
  score_list = actor_rate(casting_list, title, ci)
  inter_actor['actor_score'] = score_list
  casting = inter_actor.sort_values(by=['actor_score'], ascending=False)
  casting = casting[['Title', 'actor_score']]
  casting = casting.reset_index(drop=True)

  cast = {}
  for i in range(len(casting)):
    key = casting['Title'][i]
    value = casting['actor_score'][i]
    cast[key] = value
    castscore_df = pd.DataFrame(cast, index = ['CastScore']).T
    castscore_df[:] = scaler.fit_transform(castscore_df[:])

  return castscore_df

In [69]:
## 연극
def plot_genre_calc(title):
  plot_genre = semantic_search(title)
  if plot_genre is not None:
    plot_genre_weight = plot_genre[['Title', 'Score']]
    plot_genre_weight = plot_genre_weight.set_index(keys=['Title'])
    plot_genre_weight[:] = scaler.fit_transform(plot_genre_weight[:])

    return plot_genre_weight
  else:
    # plot_genre가 None인 경우에 대한 처리
    print("검색 결과를 찾을 수 없습니다.")
    return None

In [70]:
## 뮤지컬
def plot_genre_calc_1(title):
  plot_genre = semantic_search_1(title)
  if plot_genre is not None:
    plot_genre_weight = plot_genre[['Title', 'Score']]
    plot_genre_weight = plot_genre_weight.set_index(keys=['Title'])
    plot_genre_weight[:] = scaler.fit_transform(plot_genre_weight[:])

    return plot_genre_weight
  else:
    # plot_genre가 None인 경우에 대한 처리
    print("검색 결과를 찾을 수 없습니다.")
    return None

In [71]:
def merge_calc(weighted_df, castscore_df, plot_genre_df, pm = 0):
  func_list = []
  play_w = pd.concat([weighted_df, castscore_df, plot_genre_df], axis=1, join='inner')
  for row in play_w.iterrows():
    #print(row[1]['Weight'])
    ss = row[1]['StarScore']
    cs = row[1]['CastScore']
    pg = row[1]['Score']
    if (pm == 0): # 연극일 경우
      func = pg * 0.65 + cs * 0.2 + ss * 0.15
    else:
      func = cs * 0.35 + pg * 0.5 + ss * 0.15
    func_list.append([row[0], func])
    print(f"Title: {row[0]}, Weighted Rating: {func}")
  return func_list

In [72]:
def play_score(title):
  weighted_df = weight_calc(play_weighted_ratings)
  castscore_df = castscore_calc(title, pt_actor_dic, play_inter_dic, pt_actor_dic_t, play_ci, play_inter_actor)
  plot_genre_df = plot_genre_calc(title)
  func_list = merge_calc(weighted_df, castscore_df, plot_genre_df)
  play_score = pd.DataFrame(func_list, columns=['Title', 'Score']).sort_values('Score', ascending=False)
  return play_score

In [73]:
def musical_score(title):
  weighted_df = weight_calc(musical_weighted_ratings)
  castscore_df = castscore_calc(title, mt_actor_dic, mus_inter_dic, mt_actor_dic_t, mus_ci, mus_inter_actor)
  plot_genre_df = plot_genre_calc_1(title)
  func_list = merge_calc(weighted_df, castscore_df, plot_genre_df, pm=1)
  musical_score = pd.DataFrame(func_list, columns=['Title', 'Score']).sort_values('Score', ascending=False)
  return musical_score

In [77]:
title = "뮤지컬 〈문스토리〉"
musical_score(title).head(10)
# 0개 겹침



    TOP RESULTS

Title: 난타(NANTA) - 명동공연, Weighted Rating: 0.2366255975405214
Title: 더 맨 얼라이브 〈초이스〉, Weighted Rating: 1.0765398333193374
Title: 뮤지컬 〈22년 2개월〉, Weighted Rating: 2.55531217624176
Title: 뮤지컬 〈곤 투모로우〉, Weighted Rating: 1.1359825573939764
Title: 뮤지컬 〈구텐버그〉, Weighted Rating: 1.0431579213652271
Title: 뮤지컬 〈김종욱 찾기〉, Weighted Rating: 0.9118863681119965
Title: 뮤지컬 〈레미제라블〉 - 부산, Weighted Rating: 2.10053726742386
Title: 뮤지컬 〈레베카〉 10주년 기념공연, Weighted Rating: 2.028147420627959
Title: 뮤지컬 〈렛미플라이〉, Weighted Rating: 2.9954398025397153
Title: 뮤지컬 〈멤피스〉, Weighted Rating: 1.8490205148001206
Title: 뮤지컬 〈블랙메리포핀스〉, Weighted Rating: 2.696585346598388
Title: 뮤지컬 〈비더슈탄트〉, Weighted Rating: 1.0696236015468994
Title: 뮤지컬 〈사칠〉, Weighted Rating: 1.3781115205346977
Title: 뮤지컬 〈삼총사〉, Weighted Rating: 1.814248891336353
Title: 뮤지컬 〈셜록홈즈 : 앤더슨가의 비밀〉, Weighted Rating: 1.8997183901551025
Title: 뮤지컬 〈쇼맨_어느 독재자의 네 번째 대역배우〉, Weighted Rating: 1.571623913466388
Title: 뮤지컬 〈오페라의 유령〉 - 서울, Weighted Rating: 2.089

,Title,Score
23,뮤지컬 〈타오르는 어둠 속에서〉,3.175342
33,뮤지컬 ＇벤허＇,3.055867
8,뮤지컬 〈렛미플라이〉,2.995440
10,뮤지컬 〈블랙메리포핀스〉,2.696585
2,뮤지컬 〈22년 2개월〉,2.555312
25,뮤지컬 〈판〉,2.394302
6,뮤지컬 〈레미제라블〉 - 부산,2.100537
16,뮤지컬 〈오페라의 유령〉 - 서울,2.089696
7,뮤지컬 〈레베카〉 10주년 기념공연,2.028147
27,뮤지컬 〈후크〉,2.025566


In [ ]:
title = "연극 〈운빨로맨스〉- 대학로"
play_score(title).head(5)
# 1개 겹침

In [ ]:
title = "연극 너의 목소리가 들려"
play_score(title).head(5)
# 1개 겹침

In [ ]:
title = "뮤지컬 〈문스토리〉"
musical_score(title).head(5)
# 2개 겹침

In [ ]:
title = "뮤지컬 〈칠칠〉"
musical_score(title).head(5)
# 1개 겹침

In [ ]:
title = "뮤지컬 〈22년 2개월〉"
musical_score(title).head(5)
# 0개 겹침

# 작품테스트

In [ ]:
# 작품을 넣으면 나오는 작품들이 정말로 관련이 있는지
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(play_star, test_size=0.2, random_state=42)


In [ ]:
print(train_df.shape)
print(test_df.shape)

In [ ]:
 # True라고 예측한 것들 가운데 실제 True 비율
def get_precision(relevant, recommend):
     _intersection = set(recommend).intersection(set(relevant)) # set(recommend) & set(relevant)
     return len(_intersection) / len(recommend)

# 전체 True 가운데 True라고 예측한 비율
def get_recall(relevant, recommend):
    _intersection = set(recommend).intersection(set(relevant))
    return len(_intersection) / len(relevant)

# Precision@1부터 Precision@K 까지의 평균값
def get_avergae_precision(relevant, recommend):
    _precisions = []

    for i in range(len(recommend)):
        _recommend = recommend[:i+1]
        _precisions.append(get_precision(relevant, _recommend))

    return np.mean(_precisions)


## 기타(테스트용)

In [ ]:
for title, weighted_rating in weighted_ratings.items():
    print(f"Title: {title}, Weighted Rating: {weighted_rating}")

weighted_ratings.items()

In [ ]:
print(weighted_ratings.items())

In [ ]:
play_casting['Title'][22]

* 리스트 = [ 별점 가중치, 배우가중치 ]
* 이후 장르, 줄거리 가중치 생기면 추가해야 함

In [ ]:
import copy
play_weight = copy.deepcopy(weighted_ratings)
for title, weighted_rating in play_weight.items():
  lst = []
  for key, value in play_cast.items():
    if (title == key):
      lst.append(weighted_rating)
      lst.append(value)
      play_weight[title] = lst

play_weight

In [ ]:
for key, value in play_weight.items():
  #print(value[0], value[1])
  v = value[0] * 0.5 + value[1] * 0.1
  print(v)

In [ ]:
weighted_df = weight_calc()
castscore_df = castscore_calc("연극 〈운빨로맨스〉- 대학로")
func_list = merge_calc(weighted_df, castscore_df)
pd.DataFrame(func_list).sort_values(1, ascending=False)